# User Migration — up_users → users + user_roles

**Migration 5 of the run order** — run after geo, media, company and user-types. The canonical scripted version is `scripts/users.py` (step 5 of `scripts/migrate_data.py`); this notebook holds the same logic for interactive runs.

Migrates legacy Strapi users (users-permissions plugin) into the new `users` table, attaches profile pics / cover images to `media` rows, assigns each user a `user_roles` row from their legacy `user_type`, and links roles to `companies` from the legacy `user.company` relation.

**Read [docs/migrations/user-migration.md](../docs/migrations/user-migration.md) first** — full field mapping, dropped fields, and the password caveat.

**Prerequisites (in order):** schema migrated → `python scripts/seed.py` → geo (country lookups) → media (profile pics) → company (role↔company links) → user types (`media_usertypes_companies_migration.ipynb` or `scripts/users.py` step 1). Requires `CMS_ADMIN_EMAIL` / `CMS_ADMIN_PASSWORD` in `.env` (see fetch cell). Idempotent — safe to re-run.

⚠️ `password` is private — no API returns it (not even the admin API). `password_hash` stays NULL here; recover hashes via a direct dump of the old DB or force a password reset (see doc).

In [1]:
import os, json
from pathlib import Path

import requests
import psycopg
from psycopg.types.json import Json
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]

conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

connected to: development


## 1. Fetch legacy users — two endpoints, merged by id

The legacy CMS has a **customized** `/api/users` controller: it returns flat objects with profile fields and populated relations (`user_type`, `country`, `company`, `profilePic`, `coverImage`), but it **strips `email`, `username`, `provider`, `fbId`, `confirmed`, `blocked`, `isLoyaltyMember`, `isFeatured`, `tracking`** and ignores `start`/`limit` (the whole set comes back in one response).

The stripped fields are fetched through the **admin content-manager API** instead (super-admin login via `CMS_ADMIN_EMAIL` / `CMS_ADMIN_PASSWORD`), and the two records are merged by legacy id.

In [2]:
def fetch_public_users():
    # Custom controller: ignores start/limit and returns ALL users in one
    # response — profile fields + populated relations only (auth fields stripped).
    r = requests.get(f"{CMS_BASE_URL}/api/users", headers=HEADERS, timeout=120)
    r.raise_for_status()
    users = r.json()
    assert isinstance(users, list), f"unexpected body: {type(users)}"
    return users


def fetch_admin_users():
    # The public controller strips email/username/auth fields, so pull the raw
    # records through the admin content-manager API (super-admin login).
    r = requests.post(
        f"{CMS_BASE_URL}/admin/login",
        json={"email": os.environ["CMS_ADMIN_EMAIL"],
              "password": os.environ["CMS_ADMIN_PASSWORD"]},
        timeout=60,
    )
    r.raise_for_status()
    admin_headers = {"Authorization": f"Bearer {r.json()['data']['token']}"}

    out, page = [], 1
    while True:
        r = requests.get(
            f"{CMS_BASE_URL}/content-manager/collection-types/plugin::users-permissions.user",
            headers=admin_headers,
            params={"page": page, "pageSize": 100, "sort": "id:ASC"},
            timeout=60,
        )
        r.raise_for_status()
        body = r.json()
        out.extend(body["results"])
        if page >= body["pagination"]["pageCount"]:
            return out
        page += 1


public_users = fetch_public_users()
admin_by_id = {u["id"]: u for u in fetch_admin_users()}

# merge by id — admin record supplies auth/scalar fields, public payload wins
# for its populated relations (user_type, country, profilePic, coverImage, ...)
legacy_users = [{**admin_by_id.get(u["id"], {}), **u} for u in public_users]

missing_auth = [u["id"] for u in legacy_users if not u.get("email")]
print(f"fetched {len(legacy_users)} users; admin records: {len(admin_by_id)}; "
      f"still without email: {len(missing_auth)}")


fetched 3279 users; admin records: 3279; still without email: 0


## 2. Transform + upsert into `users`

Mapping highlights (full table in the doc):
`confirmed→email_verified`, `blocked→is_blocked`, `provider→auth_provider`, `fbId→provider_id`, `isLoyaltyMember→tier` (star_life/free), `slug→slug` (unique, collision fallback `{slug}-{legacy id}`). `profilePic` / `coverImage` / `profilePicURL` resolve to `media` rows by url — same normalization as `scripts/media.py`, so rows are reused, never duplicated. Legacy free-text `city` cannot be resolved to a `city_id` automatically — collected for manual review.

In [5]:
def split_name(u):
    first, last = u.get("firstName"), u.get("lastName")
    if not first and u.get("fullName"):
        parts = u["fullName"].strip().split(" ", 1)
        first = parts[0]
        last = last or (parts[1] if len(parts) > 1 else None)
    return first, last


def media_id_for(cur, media_obj=None, url=None):
    # Resolve a media row by url — same normalization + select-then-insert as
    # scripts/media.py, so rows migrated there are reused, never duplicated.
    if media_obj:
        url = media_obj.get("url") or url
    url = (url or "").strip()
    if not url:
        return None
    if url.startswith("/"):
        url = CMS_BASE_URL + url
    cur.execute("SELECT id FROM media WHERE url = %s", (url,))
    row = cur.fetchone()
    if row:
        return row[0]
    m = media_obj or {}
    alt = m.get("alternativeText") or m.get("caption") or m.get("name") or None
    cur.execute(
        "INSERT INTO media (url, mime_type, alt, width, height) VALUES (%s, %s, %s, %s, %s) RETURNING id",
        (url, m.get("mime"), alt, m.get("width"), m.get("height")),
    )
    return cur.fetchone()[0]


skipped, city_review = [], []
with conn.cursor() as cur:
    for u in legacy_users:
        email = (u.get("email") or "").strip().lower()
        username = (u.get("username") or "").strip() or (email.split("@")[0] if email else None)
        if not email or not username:
            skipped.append((u["id"], u.get("username"), u.get("email")))
            continue

        # username is unique — if another account already holds it, disambiguate
        cur.execute("SELECT 1 FROM users WHERE username = %s AND email <> %s", (username, email))
        if cur.fetchone():
            username = f"{username}-{u['id']}"

        # slug: legacy uid, unique — same collision guard, empty string -> NULL
        slug = (u.get("slug") or "").strip() or None
        if slug:
            cur.execute("SELECT 1 FROM users WHERE slug = %s AND email <> %s", (slug, email))
            if cur.fetchone():
                slug = f"{slug}-{u['id']}"

        first, last = split_name(u)
        profile_pic_id = media_id_for(cur, u.get("profilePic"), u.get("profilePicURL"))
        cover_image_id = media_id_for(cur, u.get("coverImage"))

        country = u.get("country") or {}
        country_name = (country.get("name") or "").strip() or None

        cur.execute(
            """
            INSERT INTO users (username, email, slug, first_name, last_name,
                               auth_provider, provider_id, email_verified, is_blocked,
                               profile_pic_id, cover_image_id, bio, social, seo, tracking,
                               is_featured, priority, tier,
                               country_id)
            VALUES (%s, %s, %s, %s, %s,
                    %s, %s, %s, %s,
                    %s, %s, %s, %s, %s, %s,
                    %s, %s, %s,
                    (SELECT id FROM countries WHERE name = %s))
            ON CONFLICT (email) DO UPDATE
            SET username       = EXCLUDED.username,
                slug           = EXCLUDED.slug,
                first_name     = EXCLUDED.first_name,
                last_name      = EXCLUDED.last_name,
                auth_provider  = EXCLUDED.auth_provider,
                provider_id    = EXCLUDED.provider_id,
                email_verified = EXCLUDED.email_verified,
                is_blocked     = EXCLUDED.is_blocked,
                profile_pic_id = EXCLUDED.profile_pic_id,
                cover_image_id = EXCLUDED.cover_image_id,
                bio            = EXCLUDED.bio,
                social         = EXCLUDED.social,
                seo            = EXCLUDED.seo,
                tracking       = EXCLUDED.tracking,
                is_featured    = EXCLUDED.is_featured,
                priority       = EXCLUDED.priority,
                tier           = EXCLUDED.tier,
                country_id     = EXCLUDED.country_id
            """,
            (
                username, email, slug, first, last,
                u.get("provider"), u.get("fbId"), bool(u.get("confirmed")), bool(u.get("blocked")),
                profile_pic_id, cover_image_id, u.get("bio"),
                Json(u["social"]) if u.get("social") else None,
                Json(u["seo"]) if u.get("seo") else None,
                Json(u["tracking"]) if u.get("tracking") else None,
                bool(u.get("isFeatured")), u.get("priority") or 0,
                "star_life" if u.get("isLoyaltyMember") else "free",
                country_name,
            ),
        )

        if u.get("city"):
            city_review.append((email, u["city"]))

conn.commit()
print(f"upserted users; skipped (no email/username): {len(skipped)}")
print(f"MANUAL REVIEW — free-text city to resolve to city_id: {len(city_review)}")
city_review[:10]


upserted users; skipped (no email/username): 0
MANUAL REVIEW — free-text city to resolve to city_id: 52


[('xaibo9@hotmail.com', 'Waterof'),
 ('ianmunyankindi@gmail.com', 'Kigali '),
 ('ayoolaa153@gmail.com', 'Lagos '),
 ('naba18thapa@gmail.com', 'Kathmandu'),
 ('jainashah07@gmail.com', 'Bhilai'),
 ('jacquelinegromada@gmail.com', 'Charleston '),
 ('rksabelhaus@gmail.com', 'Benicia'),
 ('hacklord0101@gmail.com', 'Dubai'),
 ('alorms@gmail.com', 'Lisboa'),
 ('monicagarciacruz@gmail.com', 'Porto ')]

In [6]:
# The new users table doesn't store the legacy Strapi id, but the circles /
# memories / content migrations reference users BY that id — save the mapping.
id_map = {}
with conn.cursor() as cur:
    for u in legacy_users:
        email = (u.get("email") or "").strip().lower()
        if not email:
            continue
        cur.execute("SELECT id FROM users WHERE email = %s", (email,))
        row = cur.fetchone()
        if row:
            id_map[u["id"]] = row[0]

out = ROOT / "legacy_user_id_map.json"
out.write_text(json.dumps({str(k): str(v) for k, v in id_map.items()}, indent=2))
print(f"saved {len(id_map)} legacy→new user id mappings to {out}")


saved 3279 legacy→new user id mappings to c:\Users\ReTechie\Desktop\postcard\postcard-migration\legacy_user_id_map.json


## 3. Assign roles — legacy `user_type` → `user_roles` (+ company link)

Legacy slugs are matched against the migrated `user_types` (run the user-types migration first); users without a legacy user_type get the default type (`regular`, the legacy `isDefault` row).

The legacy `user.company` N:1 relation becomes `user_roles.company_id`, matched against `companies.name` — which is what `scripts/company.py` builds its slug from, so companies must already be migrated. Note the dependency direction: the FK lived **on the user** in the legacy schema (`company.users` was just the inverse view), and in the new schema the link lives on `user_roles` — so companies are migrated **before** users, never after.

In [ ]:
conn.rollback()  # clear any open/aborted transaction from a previous failed run

with conn.cursor() as cur:
    cur.execute("SELECT slug, id FROM user_types")
    type_by_slug = dict(cur.fetchall())
    cur.execute("SELECT id FROM user_types WHERE is_default LIMIT 1")
    default_row = cur.fetchone()
    # companies are keyed by slugified name in company.py — match on trimmed name
    cur.execute("SELECT TRIM(name), id FROM companies ORDER BY id DESC")
    company_by_name = dict(cur.fetchall())  # DESC + dict -> smallest id wins on dupes

assert type_by_slug, ("user_types is EMPTY — run the user-types migration "
                      "(scripts/users.py step 1 or media_usertypes_companies_migration.ipynb) first")
assert default_row, "user_types has no is_default row — check the user-types migration"
default_type_id = default_row[0]

unmatched_types, unmatched_companies = set(), set()
with conn.cursor() as cur:
    for u in legacy_users:
        email = (u.get("email") or "").strip().lower()
        if not email:
            continue

        ut = u.get("user_type") or {}
        type_slug = (ut.get("slug") or "").strip().lower()
        type_id = type_by_slug.get(type_slug) if type_slug else default_type_id
        if type_id is None:  # legacy slug with no match -> default, and report it
            unmatched_types.add(type_slug)
            type_id = default_type_id

        company_id = None
        company_name = ((u.get("company") or {}).get("name") or "").strip()
        if company_name:
            company_id = company_by_name.get(company_name)
            if company_id is None:
                unmatched_companies.add(company_name)

        cur.execute(
            """
            INSERT INTO user_roles (user_id, user_type_id, company_id, status)
            SELECT us.id, %s, %s, 'active' FROM users us
            WHERE us.email = %s
              AND NOT EXISTS (
                    SELECT 1 FROM user_roles ur
                    WHERE ur.user_id = us.id AND ur.user_type_id = %s)
            """,
            (type_id, company_id, email, type_id),
        )
        if company_id:  # fill company on roles created by an earlier run
            cur.execute(
                """
                UPDATE user_roles ur SET company_id = %s
                FROM users us
                WHERE us.id = ur.user_id AND us.email = %s
                  AND ur.user_type_id = %s AND ur.company_id IS NULL
                """,
                (company_id, email, type_id),
            )
conn.commit()
print(f"roles assigned; unmatched legacy type slugs (fell back to default): {unmatched_types or 'none'}")
print(f"legacy companies not found in companies table: {unmatched_companies or 'none'}")

## 4. Verify

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM users")
    print("users     :", cur.fetchone()[0])
    cur.execute(
        """SELECT ut.slug, COUNT(*) FROM user_roles ur
           JOIN user_types ut ON ut.id = ur.user_type_id
           GROUP BY ut.slug ORDER BY 2 DESC"""
    )
    for slug, n in cur.fetchall():
        print(f"role {slug:15}: {n}")
    cur.execute("SELECT COUNT(*) FROM user_roles WHERE company_id IS NOT NULL")
    print("roles linked to a company:", cur.fetchone()[0])
    cur.execute("SELECT COUNT(*) FROM users WHERE profile_pic_id IS NOT NULL")
    print("users with profile pic   :", cur.fetchone()[0])
    cur.execute("SELECT COUNT(*) FROM users WHERE cover_image_id IS NOT NULL")
    print("users with cover image   :", cur.fetchone()[0])
    cur.execute("SELECT COUNT(*) FROM users WHERE password_hash IS NULL")
    print("users without password_hash (expected — see doc):", cur.fetchone()[0])
conn.close()